# 37 — Allen–Cahn: cinco métodos de colocación

**Implementado, sin ejecutar.** Cincuenta corridas: cinco métodos × diez semillas × 100000 pasos.

PDE: $u_t=0.0001u_{xx}+5(u^3-u)$. Estado inicial $x^2\cos(\pi x)$ y fronteras −1. Transformación $u=u_0+t(1-x^2)N_\theta$. Se conserva el signo positivo escrito en el artículo y verificado en el piloto; no se cambia a la reacción opuesta ni se importan datos incompatibles.

Dominio común: x ∈ [−1,1], t ∈ [0,1]. La referencia numérica propia usa diferencias centrales y BDF, con 512 y 1024 intervalos; se exige diferencia relativa entre mallas menor a 2 %. Es un control de consistencia, no una cota rigurosa de error. Los archivos del piloto no se modifican.

[Protocolo original QR-DEIM](https://arxiv.org/html/2501.07700v4#S2.SS3). Es una comparación propia en JAX, no reproducción exacta de sus resultados.

## Protocolo congelado

Cinco métodos: Random fijo, Hammersley fijo desplazado por semilla, Random-R cada 1000 pasos, RAD (k=2, c=0, 10000 candidatos nuevos) y PCA + media (4096 candidatos fijos, 20 snapshots cada 50 pasos, varianza centrada 95 %, media incluida, QR directo y complemento uniforme). Se conserva el método inicial: no hay ampliación log-det ni presupuesto guiado artificial de 200 puntos.

Red de cinco capas ocultas × 64 neuronas tanh; 2000 puntos activos; pérdida solo PDE con condiciones IC/BC duras. Adam inicial 0.001, coseno durante 100000 actualizaciones totales, sin añadir el piloto ni un warmup extra. Actualizaciones de los selectores en pasos 1000–99000; no se seleccionan puntos que no vayan a entrenarse. Las semillas 3201–3210 están separadas del piloto 3101 y los métodos comparten pesos iniciales y conjuntos independientes de validación y evaluación por semilla. Hammersley tiene sus propios puntos iniciales; los otros cuatro parten del mismo conjunto uniforme.

La comparación principal usa el menor MSE residual de validación entre los pasos 0 y 100000 observados cada 250; no el menor error de prueba. Se conservan también resultados terminales. Prueba: rejilla 256×100, sin entrenamiento supervisado. El residuo de evaluación usa puntos independientes de validación. Se guardan error L2 relativo, RMSE, residuo RMS, tiempos y resultados individuales.

Los tiempos observados incluyen compilación JAX, observación y monitores y excluyen preparar referencias y evaluar prueba; no son una medida aislada del costo del selector. Doce contrastes PCA frente a los cuatro controles en tres PDE: intervalos bootstrap pareados del 95 % con 20000 remuestreos, sin corrección por multiplicidad; no prueban un ganador global. Medianas y modelo terminal son controles descriptivos.


In [ ]:
from pathlib import Path
import sys, os, json, subprocess
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['PYTHONIOENCODING'] = 'utf-8'
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
config = json.loads((ROOT / 'pca_collocation/configs/benchmark_full.json').read_text(encoding='utf-8'))
OUT = ROOT / 'pca_collocation/results/benchmark' / config['run_id']
COMMAND = [sys.executable, '-u', str(ROOT / 'scripts/run_collocation_benchmark.py')]
print('Python:', sys.executable)
print('Resultados:', OUT)
print('Semillas:', config['seeds'])


## Ejecutar las cincuenta corridas de Allen–Cahn

La siguiente celda sí inicia el entrenamiento completo cuando la ejecutes. Usa el kernel del entorno del proyecto. Puede tardar horas en CPU. Desde terminal se obtiene el mismo resultado con `python -u scripts/run_collocation_benchmark.py --problem allen_cahn`. No ejecutar aquí y en terminal a la vez.

In [ ]:
subprocess.run(COMMAND + ['--problem', 'allen_cahn'], cwd=ROOT, check=True)


## Archivos, interrupciones y revisión

Cada método/seed guarda metrics.csv, trace.csv, constraints.json, completed.json, active_points.npz y pesos del mejor modelo de validación. Los estados terminales y de pasos 0/25000/50000/75000/100000 incluyen momentos de Adam. PCA conserva las cien historias residuales, incluyendo el tramo tardío, y el registro de rangos.

Se reutilizan funciones probadas en el piloto, pero los entrenamientos completos aún no se han ejecutado ni validado empíricamente. Repetir el mismo comando omite corridas marcadas completas y reproduce desde el inicio las incompletas; los checkpoints intermedios son para diagnóstico, no restauración automática. No ejecutar simultáneamente esta celda y el mismo problema desde terminal. Se recomienda ejecutar los tres problemas en secuencia en este equipo.

La consola se guarda en execution.log; status.json se actualiza cada observación, y last_error.txt conserva errores. Un estado running guardado no demuestra que el proceso siga activo tras un cierre abrupto. El reporte final audita las 50 corridas, lotes, restricciones, selección por validación y reconstrucciones QR tempranas y tardías.

Para regenerar únicamente el reporte, añadir --report-only al comando del problema. Si falta una corrida, el reporte falla sin entrenar. Las salidas principales permanecen dentro del repositorio, listas para revisarlas en este chat. No se publican automáticamente.


## Leer resultados ya guardados

La celda siguiente solo lee archivos. No vuelve a entrenar ni recalcula la selección del mejor modelo. La versión del reporte generada por el script contiene las mismas tablas y figuras incrustadas.

In [ ]:
from IPython.display import Image, display
import pandas as pd
RESULTS = OUT / 'allen_cahn'
assert (RESULTS / 'report_complete.json').exists(), 'Falta completar entrenamiento/reporte'
display(pd.read_csv(RESULTS / 'summary.csv'))
print((RESULTS / 'paired_comparisons.json').read_text(encoding='utf-8'))
for name in ('error_comparison.png', 'validation_curves.png'):
    display(Image(filename=str(RESULTS / name), format='png', embed=True))


## Agregación posterior

El resultado está en `pca_collocation/results/benchmark/three_pdes_five_selectors_10seeds_100k_v1/allen_cahn/`. Al finalizar este comando se genera `37_allen_cahn_benchmark_results.ipynb` con imágenes incrustadas. Cuando hayan terminado los notebooks 35, 36 y 37, ejecutar el 38; no mezclará resultados incompletos ni incluirá la semilla del piloto.